## Social dimension: data cleaning

In this notebook, we will clean our input datasets, finally merge them into a single dataframe that we save as a csv file to later on extract and calculate only the features we need in the `social_data_final_features.ipynb` notebook.
The input datasets contain data that was downloaded or received via email from the "Amt für Statistik Berlin Brandenburg" and the "Senatsverwaltung für Stadtentwicklung, Bauen und Wohnen" and manually preprocessed.
The preprocessing steps can be traced in the `processing_of_files_social_dim.csv` file in the data/social/raw directory.
In addition we will explore the data a bit by looking at missing values and the reliability of the data.

In [ ]:
# import libraries for data cleaning and file handling

import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# define path to data folder inside github repo

DATA_DIR = Path("../../data/social/raw")

# note: the different years in the key:value pairs in this dictionary are on purpose and no mistake.

files = {
    "population_2020": DATA_DIR / "population_2021.csv",
    "population_2021": DATA_DIR / "population_2022.csv",
    "population_2022": DATA_DIR / "population_2023.csv",
    "population_2023": DATA_DIR / "population_2024.csv",
    "population_2024": DATA_DIR / "population_2025.csv",
    "population_2025": DATA_DIR / "population_2026.csv",
    "fluctuation_2020": DATA_DIR / "fluctuation_2021.csv",
    "fluctuation_2021": DATA_DIR / "fluctuation_2022.csv",
    "fluctuation_2022": DATA_DIR / "fluctuation_2023.csv",
    "fluctuation_2023": DATA_DIR / "fluctuation_2024.csv",
    "fluctuation_2024": DATA_DIR / "fluctuation_2025.csv",
    "fluctuation_2025": DATA_DIR / "fluctuation_2026.csv",
    "median_income_2023": DATA_DIR / "median_income_2024.csv",
    "households_2020": DATA_DIR / "households_2021.csv",
    "households_2021": DATA_DIR / "households_2022.csv",
    "households_2022": DATA_DIR / "households_2023.csv",
    "households_2023": DATA_DIR / "households_2024.csv",
    "households_2024": DATA_DIR / "households_2025.csv",
    "mss_index_2020": DATA_DIR / "mss_index_2021.csv",
    "mss_index_2022": DATA_DIR / "mss_index_2023.csv",
    "mss_index_2024": DATA_DIR / "mss_index_2025.csv",
    "mss_context_2020": DATA_DIR / "mss_context_2021.csv",
    "mss_context_2022": DATA_DIR / "mss_context_2023.csv"
}

In [ ]:
# check whether all files exist

for name, path in files.items():
    print(name, path.exists())

In [ ]:
# before we use a function, let's check with one example, if the files are read correctly =)

df_check = pd.read_csv(files["population_2020"], delimiter=";")
df_check.head()

In [ ]:
# checking, if column names are correct

df_check.columns

In [ ]:
# checking for data types

df_check.info()

In [ ]:
df_check.dtypes

In [ ]:
# looking at specific columns to see how we need to clean the data 

df_check[["plr_key", "res_count", "age_1", "date"]].head(10)

# we can see that plr_key does NOT always have 8 digits, also numbers > 1000 are displayed as "1 000" and the "date" column does not have the data type datetime, but is displayed correctly

In [ ]:
# let's try a couple of fixes, see if they work, before we build a whole function for cleaning each file individually

# first, we check if we can fix the plr_key length; it's supposed to have 8 digits, and if it doesn't have 8 digits, we need to add a "0" at the beginning

# we can convert the plr_key column into a string while reading the file:

df_check2 = pd.read_csv(files["population_2020"], delimiter = ";", dtype={"plr_key" : "string"})

In [ ]:
df_check2.dtypes

In [ ]:
# and now fill the string with a "0" at the beginning

df_check2["plr_key"] = df_check2["plr_key"].str.zfill(8)
df_check2["plr_key"].head(10)

# seems to have worked =D

In [ ]:
# check if all values have length 8 now

df_check2["plr_key"].str.len().value_counts()

# yay! so we can fix this using the function above =)

In [ ]:
# next try: fix "date" column: convert into datetime!

df_check2["date"] = pd.to_datetime(df_check2["date"], dayfirst=True)
df_check2["date"].dtype

In [ ]:
df_check2["date"].head(9)

# WORKED! so we can also use this for our cleaning function =)

In [ ]:
# now let's try clean numbers that are > 1000 but displayed with a space between

# pandas reads them as objects, so we can treat them accordingly =D

df_check2["res_count"] = df_check2["res_count"].str.strip().str.replace(" ", "", regex=False)

In [ ]:
# now convert object into integer

# before we can do that, we need to replace "-" in the column with nan (these are missing values)

df_check2["res_count"] = df_check2["res_count"].replace(["–"], np.nan)

# and then convert

df_check2["res_count"] = pd.to_numeric(df_check2["res_count"])

# works =D

In [ ]:
# now let's look at the other files, see if we need to pay attention to something else as well:

df_check3 = pd.read_csv(files["fluctuation_2020"], delimiter = ";", dtype={"plr_key" : "string"})

In [ ]:
df_check3.head(20)

In [ ]:
df_check3.dtypes

In [ ]:
# we'll check if there are symbols for missing data as well; dtypes gives us a hint at "there should be none", since otherwise they would be read as "object" columns, not as "int", but let's still check :P

df_check3.astype("string").isin(["*", "-", "–", ""]).sum()

# nothing to see here, so let's continue with the next file :P

In [ ]:
# let's check the median_income file

df_check4 = pd.read_csv(files["median_income_2023"], delimiter = ";", dtype={"plr_key" : "string"})

In [ ]:
df_check4.head()

# we can see that the percentage has a comma instead of a period, we need to keep that in mind and fix it

In [ ]:
df_check4.dtypes

In [ ]:
df_check4.astype("string").isin(["*", "-", "–", "", "."]).sum()

# we also have symbols for missing values in here!

In [ ]:
# let's try and fix the comma issue:

df_check4["lw_ft_employee_ss_perc"] = df_check4["lw_ft_employee_ss_perc"].str.replace(",", ".", regex=False)

df_check4.head()

# worked =)

In [ ]:
# let's continue with the household files:

df_check5 = pd.read_csv(files["households_2022"], delimiter = ";", dtype={"plr_key" : "string"})

df_check5


In [ ]:
# here we definitely have the same "comma instead of period" problem, as well as the "space" problem with numbers larger than 999. Let's check for symbols for missing values

df_check5.astype("string").isin(["*", "-", "–", "", "."]).sum()

# yupp. lots.

In [ ]:
# let's check mss_index now:

df_check6 = pd.read_csv(files["mss_index_2022"], delimiter = ";", dtype={"plr_key" : "string"})

df_check6.head()

# we have the same decimal comma issue and also the numbers should be rounded! let's keep that in mind for our cleaning function

In [ ]:
# let's check mss_context files:

df_check7 = pd.read_csv(files["mss_context_2022"], delimiter = ";", dtype={"plr_key" : "string"})

df_check7.head()

# same issues as with the index file, in addition we have completely empty columns

In [ ]:
# OK, let's write the cleaning function- first, for each file individually:

def clean_single_table(file_path):
    df = pd.read_csv(
        file_path,
        delimiter=";",
        dtype={"plr_key": "string"}
    )

    # clean column names
    df.columns = (
        df.columns
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    # clean plr_key and date columns
    df["plr_key"] = df["plr_key"].str.zfill(8)
    df["date"] = pd.to_datetime(df["date"], dayfirst=True)

    # define columns that should not be converted to numeric (to define numeric columns)
    non_num_cols = ["plr_key", "plr_name", "date"]

    num_cols = [
        col for col in df.columns
        if col not in non_num_cols
    ]

    # clean and convert numeric columns
    for col in num_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace(["*", "-", "–", "", ".", "•"], np.nan)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False)
        )

        df[col] = pd.to_numeric(df[col])

    # round float columns
    float_cols = df.select_dtypes(include="float").columns
    df[float_cols] = df[float_cols].round(3)

    return df

In [ ]:
# let's test the function =D

test_clean = clean_single_table(files["households_2023"])

test_clean.head()

# looks pretty good =D

In [ ]:
test_clean.dtypes

In [ ]:
# let's go through all tables now =)

# we already have the dictionary "files", and can access both, keys and values, with .items(), so we can use that here and save the result in a new dictionary "cleaned_dfs":

cleaned_dfs = {}

for name, path in files.items():
    cleaned_dfs[name] = clean_single_table(path)
    print(f"{cleaned_dfs[name].shape}") # prints names and shapes of cleaned dfs to check =)

In [ ]:
# let's do a quick quality check:

for name, df in cleaned_dfs.items():
    print(name)
    print("shape:", df.shape)
    print("plr_key lengths:")
    print(df["plr_key"].str.len().value_counts())   # checks, if the plr_keys are right now (= have 8 digits)
    print("duplicates plr_key + date:", df.duplicated(subset=["plr_key", "date"]).sum())    # checks for duplicates in plr_keys plus date
    print()

In [ ]:
# and now: a quality report for all cleaned tables, cause we looove quality :P

quality_rows = []

for name, df in cleaned_dfs.items():
    n_rows = df.shape[0]
    n_cols = df.shape[1]

    duplicate_key_date_count = df.duplicated(subset=["plr_key", "date"]).sum()

    plr_key_min_length = df["plr_key"].str.len().min()
    plr_key_max_length = df["plr_key"].str.len().max()

    missing_total = df.isna().sum().sum()
    missing_share = df.isna().sum().sum() / df.size

    empty_columns = df.columns[df.isna().all()].tolist()

    quality_rows.append({
        "table_name": name,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "expected_542_rows": n_rows == 542,
        "duplicate_plr_key_date_count": duplicate_key_date_count,
        "plr_key_min_length": plr_key_min_length,
        "plr_key_max_length": plr_key_max_length,
        "plr_key_all_8_digits": (plr_key_min_length == 8) and (plr_key_max_length == 8),
        "missing_total": missing_total,
        "missing_share": round(missing_share, 3),
        "empty_columns_count": len(empty_columns),
        "empty_columns": empty_columns
    })

quality_report = pd.DataFrame(quality_rows)

quality_report

In [ ]:
# only show tables with potential quality issues:

quality_report[
    (quality_report["expected_542_rows"] == False)
    | (quality_report["duplicate_plr_key_date_count"] > 0)
    | (quality_report["plr_key_all_8_digits"] == False)
    | (quality_report["empty_columns_count"] > 0)
]

In [ ]:
# let's further inspect those tables:

for table_name in ["mss_index_2020", "mss_context_2020", "mss_context_2022"]:
    print(f"\n{table_name}")
    missing_by_column = cleaned_dfs[table_name].isna().sum()
    display(missing_by_column[missing_by_column > 0].sort_values(ascending=False))

Note:
Some MSS indicators are fully missing in earlier years because hey were not yet part of the MSS indicator system (mss context file) or not available in the published data (e.g. index file 2021, where S2/D2 - children in single parents households - are missing because of missing data in the years (covid?)).
We keep those columns for now and decide later to drop them or not.

In [ ]:
# let's also check for data types after cleaning:

dtype_rows = []

for table_name, df in cleaned_dfs.items():
    for col in df.columns:
        dtype_rows.append({
            "table_name": table_name,
            "column_name": col,
            "dtype": str(df[col].dtype)
        })

dtype_report = pd.DataFrame(dtype_rows)

dtype_report

In [ ]:
# only show columns that are still object or string dtype after cleaning

dtype_report[dtype_report["dtype"].astype(str).isin(["object", "string", "str"])]

# only plr_name and pr_key columns! perfect =D

In [ ]:
# now let's bring the data tables together for each "topic":

population_all = pd.concat(
    [
        cleaned_dfs["population_2020"],
        cleaned_dfs["population_2021"],
        cleaned_dfs["population_2022"],
        cleaned_dfs["population_2023"],
        cleaned_dfs["population_2024"],
        cleaned_dfs["population_2025"],
    ],
    ignore_index=True
)

fluctuation_all = pd.concat(
    [
        cleaned_dfs["fluctuation_2020"],
        cleaned_dfs["fluctuation_2021"],
        cleaned_dfs["fluctuation_2022"],
        cleaned_dfs["fluctuation_2023"],
        cleaned_dfs["fluctuation_2024"],
        cleaned_dfs["fluctuation_2025"]
    ],
    ignore_index=True
)

households_all = pd.concat(
    [
        cleaned_dfs["households_2020"],
        cleaned_dfs["households_2021"],
        cleaned_dfs["households_2022"],
        cleaned_dfs["households_2023"],
        cleaned_dfs["households_2024"],
    ],
    ignore_index=True
)

mss_index_all = pd.concat(
    [
        cleaned_dfs["mss_index_2020"],
        cleaned_dfs["mss_index_2022"],
        cleaned_dfs["mss_index_2024"],
    ],
    ignore_index=True
)

mss_context_all = pd.concat(
    [
        cleaned_dfs["mss_context_2020"],
        cleaned_dfs["mss_context_2022"],
    ],
    ignore_index=True
)

median_income_all = cleaned_dfs["median_income_2023"].copy()

In [ ]:
# let's check what we did:

print("population_all:", population_all.shape)
print("fluctuation_all:", fluctuation_all.shape)
print("households_all:", households_all.shape)
print("mss_index_all:", mss_index_all.shape)
print("mss_context_all:", mss_context_all.shape)
print("median_income_all:", median_income_all.shape)

# seems to be right =)

Now onto the next problem before we do the final merge: the population files and the household files both contain a column for resident count in the PLR. But the numbers don't match.
According to the metadata, the household count is based on the population registered with main or secondary residence and excludes persons without own household formation, such as persons in institutions.
With the following code we will explore how much the counts differ, but we will decide to keep only the resident count from the population file, as this is more precise, counting only population registered with main residence.

In [ ]:
# Compare res_count across topic tables before dropping duplicate res_count columns

res_count_checks = []

tables_with_res_count = {
    "households": households_all,
    "mss_index": mss_index_all,
    "mss_context": mss_context_all,
}

for table_name, table in tables_with_res_count.items():
    compare_df = population_all[["plr_key", "date", "res_count"]].merge(
        table[["plr_key", "date", "res_count"]],
        on=["plr_key", "date"],
        how="inner",
        suffixes=("_population", f"_{table_name}")
    )

    compare_df["res_count_difference"] = (
        compare_df[f"res_count_{table_name}"] - compare_df["res_count_population"]
    )

    res_count_checks.append({
        "table_name": table_name,
        "n_compared_rows": len(compare_df),
        "n_differences": (compare_df["res_count_difference"] != 0).sum(),
        "max_absolute_difference": compare_df["res_count_difference"].abs().max()
    })

res_count_report = pd.DataFrame(res_count_checks)

res_count_report

In [ ]:
# Compare household res_count with population res_count across all possible dates (just to exclude the possibility of having matched them to the wrong date)

date_match_rows = []

population_dates = sorted(population_all["date"].unique())
household_dates = sorted(households_all["date"].unique())

for household_date in household_dates:
    household_tmp = households_all.loc[
        households_all["date"] == household_date,
        ["plr_key", "res_count"]
    ]

    for population_date in population_dates:
        population_tmp = population_all.loc[
            population_all["date"] == population_date,
            ["plr_key", "res_count"]
        ]

        compare_tmp = household_tmp.merge(
            population_tmp,
            on="plr_key",
            how="inner",
            suffixes=("_households", "_population")
        )

        compare_tmp["difference"] = (
            compare_tmp["res_count_households"]
            - compare_tmp["res_count_population"]
        )

        date_match_rows.append({
            "household_date": household_date,
            "population_date": population_date,
            "n_compared_rows": len(compare_tmp),
            "n_differences": (compare_tmp["difference"] != 0).sum(),
            "mean_abs_difference": compare_tmp["difference"].abs().mean(),
            "max_abs_difference": compare_tmp["difference"].abs().max(),
            "total_abs_difference": compare_tmp["difference"].abs().sum()
        })

date_match_report = pd.DataFrame(date_match_rows)

date_match_report.sort_values(
    ["household_date", "total_abs_difference"]
)

In [ ]:
best_date_matches = (
    date_match_report
    .sort_values("total_abs_difference")
    .groupby("household_date")
    .head(1)
    .sort_values("household_date")
)

best_date_matches

# we see that the dates seem to be right. We will proceed with dropping the res_count column from the household files and only keep the res_count ffrom the population files.

In [ ]:
# Ok, onto the final merge now =)

merge_keys = ["plr_key", "date"]

final_df = population_all.copy()

tables_to_merge = [
    fluctuation_all,
    households_all,
    median_income_all,
    mss_index_all,
    mss_context_all,
]

# since we have plr_name and also res_count more than once (see above), we will drop those

for table in tables_to_merge:
    table = table.drop(columns=["plr_name", "res_count"], errors="ignore")
    final_df = final_df.merge(
        table,
        on=merge_keys,
        how="outer"
    )

final_df.shape

In [ ]:
# let's look at our result xD

final_df.head()

In [ ]:
final_df[["plr_key", "plr_name", "date"]].head()

In [ ]:
final_df.duplicated(subset=["plr_key", "date"]).sum()

In [ ]:
final_df["date"].value_counts().sort_index()

In [ ]:
# Now let's clean up a bit: if there are completely empty columns, we don't need to keep them

empty_columns = final_df.columns[final_df.isna().all()].tolist()
empty_columns

In [ ]:
final_df = final_df.drop(columns=empty_columns)
final_df.head()

In [ ]:
# Nice! We have a clean dataframe now with social panel data! Let's save it to csv before we continue =)

final_df.to_csv("../../data/social/raw/social_panel_clean.csv", index=False)

In [ ]:
# Now let's check how many missing values we have per row

missing_report = final_df.isna().agg(["sum", "mean"]).T.reset_index()
missing_report.columns = ["column_name", "missing_sum", "missing_share"]
missing_report = missing_report.sort_values("missing_share", ascending=False)
missing_report.head(40)

In [ ]:
# And also: How many PLRs have less than 500 and less than 1.000 people and therefore data concerning them might be unreliable?

# less than 500:

less_than_500 = final_df.query("res_count < 500 or res_count.isna()")[['plr_key', 'plr_name', 'date', 'res_count']]

less_than_500

# we can see that only 7 PLRs have less than 500 people.

In [ ]:
# less than 1.000:

less_than_1000 = final_df.query("res_count < 1000 or res_count.isna()")[['plr_key', 'plr_name', 'date', 'res_count']]

less_than_1000

# as we can see, only the later years of the Nonnendammallee wre added to the dataframe, so it is the same 7 PLRs we noted earlier.